# 🛣️ RoadWatch — YOLOv8 Training (Direct Download, No API Keys)
Runtime: GPU → T4 via Runtime → Change runtime type

In [ ]:
# ═══ Cell 1 — Install & Mount Drive ═══
!pip install ultralytics -q

import os, sys, shutil, glob, random, yaml, zipfile, xml.etree.ElementTree as ET
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUT   = '/content/drive/MyDrive/roadwatch'
EXTRACT_DIR = '/content/rdd2022'
YAML_PATH   = f'{EXTRACT_DIR}/data.yaml'

os.makedirs(DRIVE_OUT,   exist_ok=True)
os.makedirs(EXTRACT_DIR, exist_ok=True)

print('✅ Setup complete')

In [ ]:
# ═══ Cell 2 — Direct Download RDD2022 from GitHub Releases ═══

BASE = 'https://github.com/sekilab/RoadDamageDetector/releases/download/dataset3.0'
COUNTRIES = ['India', 'Japan']

for country in COUNTRIES:
    dest = f'/content/raw_{country}'
    if os.path.exists(dest):
        print(f'ℹ️  {country} already extracted')
        continue
    url = f'{BASE}/RDD2022_{country}.zip'
    zip_path = f'/content/{country}.zip'
    print(f'⬇️  Downloading {country} ...')
    !wget -q --show-progress --tries=3 --timeout=60 "{url}" -O "{zip_path}"
    print(f'📦 Extracting {country} ...')
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(dest)
    os.remove(zip_path)
    print(f'✅ {country} ready')

# ── Count available files ──
all_imgs = sorted(
    glob.glob('/content/raw_*/**/train/images/*.jpg', recursive=True) +
    glob.glob('/content/raw_*/**/train/images/*.jpeg', recursive=True) +
    glob.glob('/content/raw_*/**/train/images/*.png',  recursive=True)
)
all_xmls = sorted(
    glob.glob('/content/raw_*/**/train/annotations/xmls/*.xml', recursive=True)
)
print(f'\nTotal images : {len(all_imgs)}')
print(f'Total XMLs   : {len(all_xmls)}')

In [ ]:
# ═══ Cell 3 — Convert Pascal VOC XML → YOLO TXT + Build data.yaml ═══

# RDD2022 class mapping → our 4 classes
# D00=longitudinal crack, D10=transverse crack,
# D20=alligator crack,    D40=pothole, D44=pothole
RAW_CLASS_MAP = {
    'D00': 'crack',          # longitudinal
    'D01': 'crack',
    'D10': 'crack',          # transverse
    'D11': 'crack',
    'D20': 'surface_damage', # alligator / area
    'D40': 'pothole',
    'D44': 'pothole',
    'D43': 'pothole',
    'D50': 'surface_damage',
    'D0w0': 'crack',
}
CLASS_NAMES = ['pothole', 'crack', 'surface_damage', 'multiple']

def xml_to_yolo(xml_path, img_w, img_h):
    """Convert one Pascal VOC XML to list of YOLO lines."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    classes_in_image = set()
    
    for obj in root.findall('object'):
        raw_name = obj.find('name').text.strip()
        cls_name = RAW_CLASS_MAP.get(raw_name)
        if cls_name is None:
            continue
        classes_in_image.add(cls_name)
        
    # If multiple different damage types exist in the image, label them all as 'multiple' for this simplified schema.
    # Or, following the prompt "If image has >1 class type detected: label as multiple (cls_id=3)"
    # We will override the cls_id if len(classes_in_image) > 1
    is_multiple = len(classes_in_image) > 1

    for obj in root.findall('object'):
        raw_name = obj.find('name').text.strip()
        cls_name = RAW_CLASS_MAP.get(raw_name)
        if cls_name is None:
            continue
            
        if is_multiple:
            cls_id = 3 # multiple
        else:
            cls_id = CLASS_NAMES.index(cls_name)
            
        bndbox = obj.find('bndbox')
        xmin = float(bndbox.find('xmin').text)
        ymin = float(bndbox.find('ymin').text)
        xmax = float(bndbox.find('xmax').text)
        ymax = float(bndbox.find('ymax').text)
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h
        cx, cy, bw, bh = [
            round(min(max(v, 0.0), 1.0), 6) for v in [cx, cy, bw, bh]
        ]
        if bw > 0 and bh > 0:
            lines.append(f'{cls_id} {cx} {cy} {bw} {bh}')
    return lines

# ── Create YOLO folder structure ──
for split in ['train', 'val']:
    os.makedirs(f'{EXTRACT_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{EXTRACT_DIR}/labels/{split}', exist_ok=True)

# ── Build paired list (image, xml) ──
pairs = []
xml_index = {Path(x).stem: x for x in all_xmls}

for img_path in all_imgs:
    stem = Path(img_path).stem
    if stem in xml_index:
        pairs.append((img_path, xml_index[stem]))

print(f'Matched image+label pairs: {len(pairs)}')

# ── 80/20 train/val split ──
random.seed(42)
random.shuffle(pairs)
split_idx  = int(len(pairs) * 0.8)
train_pairs = pairs[:split_idx]
val_pairs   = pairs[split_idx:]

# ── Convert and copy ──
from PIL import Image
import traceback

skipped = 0
for split_name, split_pairs in [('train', train_pairs), ('val', val_pairs)]:
    for img_path, xml_path in split_pairs:
        try:
            with Image.open(img_path) as im:
                w, h = im.size
            yolo_lines = xml_to_yolo(xml_path, w, h)
            if not yolo_lines:
                skipped += 1
                continue
            fname = Path(img_path).name
            shutil.copy(img_path, f'{EXTRACT_DIR}/images/{split_name}/{fname}')
            label_path = f'{EXTRACT_DIR}/labels/{split_name}/{Path(img_path).stem}.txt'
            with open(label_path, 'w') as lf:
                lf.write('\n'.join(yolo_lines))
        except Exception:
            skipped += 1

train_count = len(glob.glob(f'{EXTRACT_DIR}/images/train/*'))
val_count   = len(glob.glob(f'{EXTRACT_DIR}/images/val/*'))
print(f'\nTrain: {train_count} images')
print(f'Val  : {val_count} images')
print(f'Skip : {skipped} (no valid boxes or read error)')

# ── Write data.yaml ──
with open(YAML_PATH, 'w') as f:
    yaml.dump({
        'path' : EXTRACT_DIR,
        'train': 'images/train',
        'val'  : 'images/val',
        'nc'   : 4,
        'names': CLASS_NAMES,
    }, f, default_flow_style=False)

print(f'\n📄 data.yaml:')
print(open(YAML_PATH).read())

In [ ]:
# ═══ Cell 4 — Train YOLOv8n ═══
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data      = YAML_PATH,
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    patience  = 10,
    mosaic    = 1.0,
    flipud    = 0.3,
    fliplr    = 0.5,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
    translate = 0.1,
    scale     = 0.5,
    degrees   = 5.0,
    project   = '/content/runs',
    name      = 'roadwatch',
    exist_ok  = True,
    device    = 0,
    workers   = 2,
    cache     = True,
    optimizer = 'AdamW',
    lr0       = 0.001,
    lrf       = 0.01,
    weight_decay = 0.0005,
    warmup_epochs = 3,
    close_mosaic  = 10,
    amp       = True,
)

BEST_PT = '/content/runs/roadwatch/weights/best.pt'
print(f'\n✅ Training complete')
print(f'Best weights: {BEST_PT}')

In [ ]:
# ═══ Cell 5 — Evaluate + Save to Drive ═══
from ultralytics import YOLO
import shutil, os

BEST_PT = '/content/runs/roadwatch/weights/best.pt'

best_model = YOLO(BEST_PT)
metrics = best_model.val(data=YAML_PATH, verbose=False)

map50    = metrics.box.map50
map5095  = metrics.box.map

print('╔══════════════════════════════════════╗')
print('║   RoadWatch — Final Evaluation       ║')
print('╠══════════════════════════════════════╣')
print(f'║  mAP@50     : {map50:.4f}                  ║')
print(f'║  mAP@50-95  : {map5095:.4f}                  ║')
print('╚══════════════════════════════════════╝')

CLASS_NAMES = ['pothole', 'crack', 'surface_damage', 'multiple']
if hasattr(metrics.box, 'maps'):
    print('\nPer-class mAP@50:')
    for i, name in enumerate(CLASS_NAMES):
        if i < len(metrics.box.maps):
            print(f'  {name:20s} → {metrics.box.maps[i]:.4f}')

# ── Accuracy gate ──
if map50 >= 0.50:
    dst = os.path.join(DRIVE_OUT, 'best.pt')
    shutil.copy(BEST_PT, dst)
    print(f'\n💾 Saved to Google Drive: {dst}')
    print('✅ Model meets minimum accuracy threshold (mAP@50 ≥ 0.50)')
else:
    print(f'\n⚠️  mAP@50 = {map50:.4f} is below 0.50')
    print('   Saving anyway — consider retraining with more epochs')
    dst = os.path.join(DRIVE_OUT, f'best_map{map50:.3f}.pt')
    shutil.copy(BEST_PT, dst)
    print(f'   Saved as: {dst}')

In [ ]:
# ═══ Cell 6 — Validate Inference on 3 Samples ═══
from ultralytics import YOLO
import glob, random, os

BEST_PT = '/content/runs/roadwatch/weights/best.pt'
best_model = YOLO(BEST_PT)

val_imgs = (
    glob.glob(f'{EXTRACT_DIR}/images/val/*.jpg') +
    glob.glob(f'{EXTRACT_DIR}/images/val/*.png')
)
random.seed(99)
samples = random.sample(val_imgs, min(3, len(val_imgs)))

CLASS_NAMES = ['pothole', 'crack', 'surface_damage', 'multiple']

print('🔍 Inference on 3 validation images:\n')
for i, img_path in enumerate(samples, 1):
    preds = best_model(img_path, conf=0.25, verbose=False)
    r = preds[0]
    n = len(r.boxes) if r.boxes is not None else 0
    print(f'━━━ Sample {i}: {os.path.basename(img_path)} ━━━')
    print(f'    Detections: {n}')
    if n == 0:
        print('    (nothing above 0.25 conf)')
    else:
        for j in range(n):
            cls  = int(r.boxes.cls[j])
            conf = float(r.boxes.conf[j])
            name = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f'class_{cls}'
            print(f'    → {name:20s}  conf={conf:.4f}')
    print()

print('✅ Inference validation complete')
print(f'\nDownload best.pt from Google Drive:')
print(f'  My Drive → roadwatch → best.pt')
print(f'\nThen set on Render:')
print(f'  YOLO_MODEL_PATH = /path/to/best.pt')